<a href="https://colab.research.google.com/github/Savidilsh/vggt/blob/chamudi_R/VGGT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git config --global user.name "chamudiR"
!git config --global user.email "chamudiransika@gmail.com"


In [5]:
!git clone https://github.com/chamudiR/vggt.git

Cloning into 'vggt'...
remote: Enumerating objects: 565, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 565 (delta 105), reused 71 (delta 66), pack-reused 395 (from 4)
Receiving objects: 100% (565/565), 70.13 MiB | 22.07 MiB/s, done.
Resolving deltas: 100% (210/210), done.


In [8]:
%cd vggt


/content/vggt


In [1]:
# Cell 1: Upload images (CLEARS OLD IMAGES FIRST)
from google.colab import files
import os

print("=" * 70)
print("UPLOAD YOUR IMAGES")
print("=" * 70)

# Check if images folder exists and has files
if os.path.exists('images') and os.listdir('images'):
    num_old = len(os.listdir('images'))
    print(f"\n⚠️ Found {num_old} old images in folder")
    print("Clearing old images...")
    !rm -rf images/*
    print("✓ Old images removed\n")
else:
    print("\n(No old images found)\n")

# Create images folder
os.makedirs('images', exist_ok=True)

print("Click 'Choose Files' below and select your NEW images")
print("(Select multiple images: Ctrl+Click or Shift+Click)")
print()

# Upload new images
uploaded = files.upload()

# Move to images folder
for filename in uploaded.keys():
    os.rename(filename, f'images/{filename}')

print(f"\n✓ Uploaded {len(uploaded)} new images successfully!")
print("\nCurrent images in folder:")
!ls -1 images/

print("\n" + "=" * 70)
print("✓ Ready to proceed to Cell 3!")
print("=" * 70)


UPLOAD YOUR IMAGES

⚠️ Found 5 old images in folder
Clearing old images...
✓ Old images removed

Click 'Choose Files' below and select your NEW images
(Select multiple images: Ctrl+Click or Shift+Click)



Saving 001.png to 001.png
Saving 002.png to 002.png
Saving 003.png to 003.png
Saving 004.png to 004.png
Saving 005.png to 005.png

✓ Uploaded 5 new images successfully!

Current images in folder:
001.png
002.png
003.png
004.png
005.png

✓ Ready to proceed to Cell 3!


In [2]:
# Cell 2: Install VGGT with correct NumPy version
print("=" * 70)
print("INSTALLING VGGT & DEPENDENCIES")
print("=" * 70)
print("\nThis takes 1-2 minutes...\n")

# Step 1: Fix NumPy version incompatibility
print("[1/4] Fixing NumPy version...")
!pip uninstall numpy scipy -y -q
!pip install numpy==1.26.4 -q
print("✓ NumPy 1.26.4 installed")

# Step 2: Install VGGT
print("\n[2/4] Installing VGGT from GitHub...")
!pip install --no-deps git+https://github.com/facebookresearch/vggt.git -q
print("✓ VGGT installed")

# Step 3: Install dependencies
print("\n[3/4] Installing dependencies...")
!pip install -q huggingface_hub pillow
print("✓ Dependencies installed")

# Step 4: Verify installation
print("\n[4/4] Verifying installation...")
import numpy as np
import torch
from vggt.models.vggt import VGGT

print(f"  ✓ NumPy: {np.__version__}")
print(f"  ✓ PyTorch: {torch.__version__}")
print(f"  ✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  ✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  ✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  ✓ VGGT: OK")

print("\n" + "=" * 70)
print("✓✓✓ INSTALLATION COMPLETE!")
print("=" * 70)
print("\nReady to run Cell 3!")


INSTALLING VGGT & DEPENDENCIES

This takes 1-2 minutes...

[1/4] Fixing NumPy version...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 109.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
librosa 0.11.0 requires scipy>=1.6.0, which is not installed.
pytensor 2.31.7 requires scipy<2,>=1, which is not installed.
yellowbrick 1.5 requires scipy>=1.0.0, which is not installed.
cuml-cu12 25.6.0 requires scipy>=1.8.0, which is not installed.
scikit-learn 1.6.1 requires scipy>=1.6.0, which is not installed.
sklearn-pandas 2.2.0 requires scipy>=1.5.1, which is not installed.
lightgbm 4.6.0 requires scipy, which is not installed.
osqp 1.0.4 requires scipy>=0.13.2, which is not installed.
statsmodels 0.14.5 requires scipy!=1.9.2,>=1.8, which is not installed.
mizan

In [2]:
# Cell 3: VGGT 3D Reconstruction
import torch
import numpy as np
from pathlib import Path
from glob import glob
import time

print("=" * 70)
print("VGGT 3D RECONSTRUCTION")
print("=" * 70)

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✓ Device: {device}")
if device == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Get images
image_paths = sorted(glob('images/*.png'))
if not image_paths:
    print("\n❌ ERROR: No images found in images/ folder!")
    print("Please run Cell 1 again to upload images.")
else:
    print(f"\n✓ Found {len(image_paths)} images")
    for i, p in enumerate(image_paths[:5], 1):
        print(f"  {i}. {Path(p).name}")
    if len(image_paths) > 5:
        print(f"  ... and {len(image_paths) - 5} more")

# Load VGGT model
print("\n" + "=" * 70)
print("LOADING VGGT MODEL")
print("=" * 70)
print("First run: Downloads ~4.8GB from HuggingFace (2-3 min)")
print("Subsequent runs: Loads from cache (~30 seconds)")
print()

start_time = time.time()

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

# Download and load model
model = VGGT.from_pretrained("facebook/VGGT-1B")
model = model.to(device).eval()

print(f"✓ Model loaded in {time.time() - start_time:.1f}s")

# Load and preprocess images
print("\n" + "=" * 70)
print("LOADING & PREPROCESSING IMAGES")
print("=" * 70)

images = load_and_preprocess_images(image_paths).to(device)
print(f"✓ Images loaded: {images.shape}")
print(f"  Format: [N={images.shape[0]}, C={images.shape[1]}, H={images.shape[2]}, W={images.shape[3]}]")

# Run inference
print("\n" + "=" * 70)
print("RUNNING INFERENCE")
print("=" * 70)
print("This takes 30-60 seconds on GPU...\n")

inference_start = time.time()

with torch.no_grad():
    # Add batch dimension
    batch = images.unsqueeze(0)
    print(f"Batch shape: {batch.shape}\n")

    # Step 1: Aggregator (Feature extraction)
    print("[1/4] Aggregator (feature extraction)...", end=" ", flush=True)
    t = time.time()
    tokens, ps_idx = model.aggregator(batch)
    print(f"✓ {time.time() - t:.1f}s")

    # Step 2: Camera head (Camera pose prediction)
    print("[2/4] Camera head (pose prediction)...", end=" ", flush=True)
    t = time.time()
    pose_enc = model.camera_head(tokens)[-1]
    extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, batch.shape[-2:])
    print(f"✓ {time.time() - t:.1f}s")
    print(f"        Extrinsics: {extrinsic.shape}, Intrinsics: {intrinsic.shape}")

    # Step 3: Depth head (Depth map prediction)
    print("[3/4] Depth head (depth prediction)...", end=" ", flush=True)
    t = time.time()
    depth_map, depth_conf = model.depth_head(tokens, batch, ps_idx)
    print(f"✓ {time.time() - t:.1f}s")
    print(f"        Depth maps: {depth_map.shape}")

    # Step 4: Unproject to 3D
    print("[4/4] Unprojecting to 3D point cloud...", end=" ", flush=True)
    t = time.time()
    point_map = unproject_depth_map_to_point_map(
        depth_map.squeeze(0),
        extrinsic.squeeze(0),
        intrinsic.squeeze(0)
    )
    print(f"✓ {time.time() - t:.1f}s")
    print(f"        Point map: {point_map.shape}")

inference_time = time.time() - inference_start
print(f"\n✓ Total inference time: {inference_time:.1f}s")

# Save point cloud
print("\n" + "=" * 70)
print("SAVING POINT CLOUD")
print("=" * 70)

# Convert to numpy if needed
if isinstance(point_map, torch.Tensor):
    points = point_map.cpu().numpy()
else:
    points = point_map

if isinstance(images, torch.Tensor):
    colors = images.cpu().numpy()
else:
    colors = images

# Reshape
N, H, W, _ = points.shape
total_points = N * H * W
points = points.reshape(-1, 3)
colors = colors.transpose(0, 2, 3, 1).reshape(-1, 3)

print(f"Total points before filtering: {total_points:,}")

# Filter valid points
valid = ~np.isnan(points).any(axis=1) & ~np.isinf(points).any(axis=1)
valid &= (np.linalg.norm(points, axis=1) < 100)

points = points[valid]
colors = (colors[valid] * 255).clip(0, 255).astype(np.uint8)

print(f"Valid points after filtering: {len(points):,} ({len(points)/total_points*100:.1f}%)")

# Save PLY file
filename = "vggt_reconstruction.ply"
print(f"\nWriting {filename}...")

with open(filename, 'w') as f:
    f.write("ply\n")
    f.write("format ascii 1.0\n")
    f.write(f"element vertex {len(points)}\n")
    f.write("property float x\n")
    f.write("property float y\n")
    f.write("property float z\n")
    f.write("property uchar red\n")
    f.write("property uchar green\n")
    f.write("property uchar blue\n")
    f.write("end_header\n")

    # Write points in chunks with progress
    chunk_size = 50000
    for i in range(0, len(points), chunk_size):
        end = min(i + chunk_size, len(points))
        for j in range(i, end):
            f.write(f"{points[j,0]:.6f} {points[j,1]:.6f} {points[j,2]:.6f} ")
            f.write(f"{colors[j,0]} {colors[j,1]} {colors[j,2]}\n")
        if i % 100000 == 0 and i > 0:
            print(f"  Progress: {i:,}/{len(points):,} points written...")

print(f"✓ Saved {filename}")

# Save camera parameters
if isinstance(extrinsic, torch.Tensor):
    np.save("camera_extrinsics.npy", extrinsic.cpu().numpy())
    np.save("camera_intrinsics.npy", intrinsic.cpu().numpy())
    np.save("depth_maps.npy", depth_map.cpu().numpy())
else:
    np.save("camera_extrinsics.npy", extrinsic)
    np.save("camera_intrinsics.npy", intrinsic)
    np.save("depth_maps.npy", depth_map)

print("✓ Saved camera parameters")

# Summary
total_time = time.time() - start_time
print("\n" + "=" * 70)
print("✓✓✓ RECONSTRUCTION COMPLETED SUCCESSFULLY! ✓✓✓")
print("=" * 70)
print(f"Images processed: {len(image_paths)}")
print(f"Points generated: {len(points):,}")
print(f"Model load time: {start_time:.1f}s")
print(f"Inference time: {inference_time:.1f}s")
print(f"Total time: {total_time:.1f}s")
print(f"\nOutput files:")
print(f"  1. vggt_reconstruction.ply ({len(points):,} points)")
print(f"  2. camera_extrinsics.npy")
print(f"  3. camera_intrinsics.npy")
print(f"  4. depth_maps.npy")
print("=" * 70)
print("\n✓ Ready to run Cell 4 to download files!")


VGGT 3D RECONSTRUCTION

✓ Device: cuda
✓ GPU: Tesla T4
✓ VRAM: 15.8 GB

✓ Found 5 images
  1. 001.png
  2. 002.png
  3. 003.png
  4. 004.png
  5. 005.png

LOADING VGGT MODEL
First run: Downloads ~4.8GB from HuggingFace (2-3 min)
Subsequent runs: Loads from cache (~30 seconds)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.03G [00:00<?, ?B/s]

✓ Model loaded in 120.9s

LOADING & PREPROCESSING IMAGES
✓ Images loaded: torch.Size([5, 3, 518, 518])
  Format: [N=5, C=3, H=518, W=518]

RUNNING INFERENCE
This takes 30-60 seconds on GPU...

Batch shape: torch.Size([1, 5, 3, 518, 518])

[1/4] Aggregator (feature extraction)... ✓ 6.3s
[2/4] Camera head (pose prediction)... ✓ 0.2s
        Extrinsics: torch.Size([1, 5, 3, 4]), Intrinsics: torch.Size([1, 5, 3, 3])
[3/4] Depth head (depth prediction)... ✓ 0.5s
        Depth maps: torch.Size([1, 5, 518, 518, 1])
[4/4] Unprojecting to 3D point cloud... ✓ 0.1s
        Point map: (5, 518, 518, 3)

✓ Total inference time: 7.1s

SAVING POINT CLOUD
Total points before filtering: 1,341,620
Valid points after filtering: 1,341,620 (100.0%)

Writing vggt_reconstruction.ply...
  Progress: 100,000/1,341,620 points written...
  Progress: 200,000/1,341,620 points written...
  Progress: 300,000/1,341,620 points written...
  Progress: 400,000/1,341,620 points written...
  Progress: 500,000/1,341,620 point

In [ ]:
# Cell 4: Download output files
from google.colab import files

print("=" * 70)
print("DOWNLOADING OUTPUT FILES")
print("=" * 70)
print("\nDownloading files to your computer...\n")

# Download all files
files.download('vggt_reconstruction.ply')
print("✓ Downloaded vggt_reconstruction.ply")

files.download('camera_extrinsics.npy')
print("✓ Downloaded camera_extrinsics.npy")

files.download('camera_intrinsics.npy')
print("✓ Downloaded camera_intrinsics.npy")

files.download('depth_maps.npy')
print("✓ Downloaded depth_maps.npy")

print("\n" + "=" * 70)
print("✓✓✓ ALL FILES DOWNLOADED!")
print("=" * 70)
print("\nVisualize your point cloud with:")
print("  • MeshLab: https://www.meshlab.net/")
print("  • CloudCompare: https://www.danielgm.net/cc/")
print("  • Online viewer: https://3dviewer.net/")
print("=" * 70)


VGGT 3D RECONSTRUCTION - Google Colab GPU

✓ Device: cuda
✓ GPU: Tesla T4
✓ VRAM: 15.8 GB

✓ Found 20 images

LOADING MODEL
✓ Model loaded in 34.3s

LOADING IMAGES
✓ Images: torch.Size([20, 3, 392, 518])

RUNNING INFERENCE
[1/4] Aggregator... ✓ 29.5s
[2/4] Camera head... ✓ 0.0s
[3/4] Depth head... ✓ 1.4s
[4/4] Unprojecting... ✓ 1.3s

✓ Total inference: 32.2s

SAVING POINT CLOUD
Total points before filtering: 4,061,120
Valid points after filtering: 4,061,120
Writing vggt_reconstruction.ply...
✓ Saved vggt_reconstruction.ply (4,061,120 points)
✓ Saved camera parameters

✓✓✓ RECONSTRUCTION COMPLETED!
Images: 20
Points: 4,061,120
Output: vggt_reconstruction.ply



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ All files downloaded!


In [3]:
# BONUS: Visualize point cloud in Colab
!pip install -q plotly

import plotly.graph_objects as go
import numpy as np

print("Loading point cloud for visualization...")

# Load PLY file
with open('vggt_reconstruction.ply', 'r') as f:
    lines = f.readlines()

header_end = lines.index('end_header\n') + 1
data = []
for line in lines[header_end:]:
    vals = line.strip().split()
    data.append([float(vals[0]), float(vals[1]), float(vals[2]),
                 int(vals[3]), int(vals[4]), int(vals[5])])

data = np.array(data)
points = data[:, :3]
colors = data[:, 3:6].astype(int)

# Subsample for performance (show every Nth point)
step = max(1, len(points) // 50000)
pts_sub = points[::step]
cols_sub = colors[::step]

print(f"Visualizing {len(pts_sub):,} points (subsampled from {len(points):,})...")

# Create RGB strings
rgb_colors = [f'rgb({c[0]},{c[1]},{c[2]})' for c in cols_sub]

# Create 3D plot
fig = go.Figure(data=[go.Scatter3d(
    x=pts_sub[:, 0],
    y=pts_sub[:, 1],
    z=pts_sub[:, 2],
    mode='markers',
    marker=dict(size=2, color=rgb_colors)
)])

fig.update_layout(
    title=f"VGGT 3D Reconstruction ({len(points):,} points)",
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data'
    ),
    width=900, height=700
)

fig.show()
print("\n✓ Use mouse to rotate/zoom the point cloud!")


Loading point cloud for visualization...
Visualizing 51,601 points (subsampled from 1,341,620)...



✓ Use mouse to rotate/zoom the point cloud!


In [9]:
!git fetch origin



In [10]:
!git checkout -b chamudiR
!git push -u origin chamudiR


Switched to a new branch 'chamudiR'
fatal: could not read Username for 'https://github.com': No such device or address
